In [11]:
from process_mobidb_scores import get_mobidb_scores
from alphasync_data import get_alphasync_data, parse_alphasync_data
from protein_region_features import features_binding_region_df, get_amino_acid_composition, calculate_scd, calculate_shd, calculate_fcr, get_sec_str_freq, get_average_feature_value
from pathlib import Path
import pandas as pd

In [2]:
# get the current directory
current_dir = Path.cwd()

# path to the input fasta file
input_fasta = current_dir.parents[1] / "data" / "raw" / "mobidb_search_2026-05-05T08-28-46.fasta"

In [38]:
# get the mobidb scores for the given input fasta file and the given score patterns
human_proteome_seq_df = get_mobidb_scores(str(input_fasta), ["derived-binding_mode_disorder_to_disorder-mobi", "derived-binding_mode_disorder_to_order-mobi", "derived-binding_mode_context_dependent-mobi"])

# print the first 5 rows of the complete data dataframe
print(human_proteome_seq_df.head())

# print the number of rows in the complete data dataframe
print(f"Number of rows in the complete data dataframe: {human_proteome_seq_df.shape[0]}")

  uniprot_id                                           sequence  \
0     P31946  MTMDKSELVQKAKLAEQAERYDDMAAAMKAVTEQGHELSNEERNLL...   
1     Q04917  MGDREQLLQRARLAEQAERYDDMASAMKAVTELNEPLSNEDRNLLS...   
2     P62258  MDDREDLVYQAKLAEQAERYDEMVESMKKVAGMDVELTVEERNLLS...   
3     P61981  MVDREQLVQKARLAEQAERYDDMAAAMKNVTELNEPLSNEERNLLS...   
4     P31947  MERASLIQKAKLAEQAERYEDMAAFMKGAVEKGEELSCEERNLLSV...   

      derived_binding_mode_disorder_to_disorder_mobi  \
0                                                NaN   
1  0000000000000000000000000000000000000000000000...   
2  0000000000000000000000000000000000000000000000...   
3                                                NaN   
4  0000000000000000000000000000000000000000000000...   

         derived_binding_mode_disorder_to_order_mobi  \
0                                                NaN   
1                                                NaN   
2  1111111111111111111111111111111100000000000000...   
3  000000000000000000000000000000000

In [39]:
# filter the dataframe to get only those that has mobidb scores for at least one of the binding mode scores
binding_mode_scores = ["derived_binding_mode_disorder_to_disorder_mobi", "derived_binding_mode_disorder_to_order_mobi", "derived_binding_mode_context_dependent_mobi"]
df_binding_idr_seq = human_proteome_seq_df.dropna(subset=binding_mode_scores, how="all").copy()

# print the number of rows in the filtered dataframe
print(f"Number of rows in the filtered dataframe: {df_binding_idr_seq.shape[0]}")

print(df_binding_idr_seq.head())

Number of rows in the filtered dataframe: 4751
  uniprot_id                                           sequence  \
1     Q04917  MGDREQLLQRARLAEQAERYDDMASAMKAVTELNEPLSNEDRNLLS...   
2     P62258  MDDREDLVYQAKLAEQAERYDEMVESMKKVAGMDVELTVEERNLLS...   
3     P61981  MVDREQLVQKARLAEQAERYDDMAAAMKNVTELNEPLSNEERNLLS...   
4     P31947  MERASLIQKAKLAEQAERYEDMAAFMKGAVEKGEELSCEERNLLSV...   
5     P27348  MEKTELIQKAKLAEQAERYDDMATCMKAVTEQGAELSNEERNLLSV...   

      derived_binding_mode_disorder_to_disorder_mobi  \
1  0000000000000000000000000000000000000000000000...   
2  0000000000000000000000000000000000000000000000...   
3                                                NaN   
4  0000000000000000000000000000000000000000000000...   
5  0000000000000000000000000000000000000000000000...   

         derived_binding_mode_disorder_to_order_mobi  \
1                                                NaN   
2  1111111111111111111111111111111100000000000000...   
3  000000000000000000000000000000000000000000

In [ ]:
# filter the complete data dataframe to get the dataframes for the three binding mode classes
#do_data_df = complete_data_df.dropna(subset=["derived_binding_mode_disorder_to_disorder_mobi"], thresh=1)
#dd_data_df = complete_data_df.dropna(subset=["derived_binding_mode_disorder_to_order_mobi"], thresh=1)
#cd_data_df = complete_data_df.dropna(subset=["derived_binding_mode_context_dependent_mobi"], thresh=1)

# save the three dataframes as csv files in the processed data folder
#processed_data_folder = current_dir.parents[1] / "data" / "processed"
#do_data_df.to_csv(processed_data_folder / "disorder_to_disorder_binding_mode_data.csv", index=False)
#dd_data_df.to_csv(processed_data_folder / "disorder_to_order_binding_mode_data.csv", index=False)
#cd_data_df.to_csv(processed_data_folder / "context_dependent_binding_mode_data.csv", index=False)

In [3]:
# path to the processed data folder
processed_data_folder = current_dir.parents[1] / "data" / "processed"


In [ ]:
# save the dataframe as csv files in the processed data folder
df_binding_idr_seq.to_csv(processed_data_folder / "binding_idr_sequences.csv", index=False)

In [4]:
# define the output data folder for the alphasync data
output_data_folder = current_dir.parents[1] / "data" / "raw" / "alphasync_data"

In [47]:
# get the alphasync data for the binding idr sequences
print("Downloading the alphasync data for the given uniprot IDs...")
get_alphasync_data(df_binding_idr_seq["uniprot_id"].tolist(), str(output_data_folder))

Failed to download A0A0A0MSA1 (Status: 404)
Failed to download F2Z2U4 (Status: 404)
Failed to download E9PG32 (Status: 404)
Failed to download A0A8V8TRG9 (Status: 404)
Failed to download A0A669KB38 (Status: 404)


In [49]:
# remove the ids that couldn't be downloaded from the alphasync data and save the filtered dataframe as a csv file in the processed data folder
remove_ids = ["A0A0A0MSA1", "F2Z2U4", "E9PG32", "A0A8V8TRG9", "A0A669KB38"]

# filter the dataframe to remove the rows with the uniprot ids in the remove_ids list
df_binding_idr_seq_filtered = df_binding_idr_seq[~df_binding_idr_seq["uniprot_id"].isin(remove_ids)].copy()
df_binding_idr_seq_filtered.to_csv(processed_data_folder / "binding_idr_sequences_filtered.csv", index=False)

In [5]:
# read the filtered dataframe from the csv file in the processed data folder
df_binding_idr_seq_filtered = pd.read_csv(processed_data_folder / "binding_idr_sequences_filtered.csv")

In [6]:
# add the alphasync data for the filtered dataframe and save the updated dataframe as a csv file in the processed data folder
df_binding_idr_seq_filtered["plddt"] = df_binding_idr_seq_filtered["uniprot_id"].apply(lambda x: parse_alphasync_data(x, "plddt", str(output_data_folder), divide_by_100=True))
df_binding_idr_seq_filtered["relasa"] = df_binding_idr_seq_filtered["uniprot_id"].apply(lambda x: parse_alphasync_data(x, "relasa", str(output_data_folder)))
df_binding_idr_seq_filtered["sec"] = df_binding_idr_seq_filtered["uniprot_id"].apply(lambda x: parse_alphasync_data(x, "sec", str(output_data_folder)))

print(df_binding_idr_seq_filtered.head())

df_binding_idr_seq_filtered.to_csv(processed_data_folder / "binding_idr_sequences_filtered_with_alphasync.csv", index=False)

  uniprot_id                                           sequence  \
0     Q04917  MGDREQLLQRARLAEQAERYDDMASAMKAVTELNEPLSNEDRNLLS...   
1     P62258  MDDREDLVYQAKLAEQAERYDEMVESMKKVAGMDVELTVEERNLLS...   
2     P61981  MVDREQLVQKARLAEQAERYDDMAAAMKNVTELNEPLSNEERNLLS...   
3     P31947  MERASLIQKAKLAEQAERYEDMAAFMKGAVEKGEELSCEERNLLSV...   
4     P27348  MEKTELIQKAKLAEQAERYDDMATCMKAVTEQGAELSNEERNLLSV...   

      derived_binding_mode_disorder_to_disorder_mobi  \
0  0000000000000000000000000000000000000000000000...   
1  0000000000000000000000000000000000000000000000...   
2                                                      
3  0000000000000000000000000000000000000000000000...   
4  0000000000000000000000000000000000000000000000...   

         derived_binding_mode_disorder_to_order_mobi  \
0                                                      
1  1111111111111111111111111111111100000000000000...   
2  0000000000000000000000000000000000000000000000...   
3                                   

In [7]:

# get features for the binding regions from that of the complete protein sequence
df_binding_region_do = features_binding_region_df(df_binding_idr_seq_filtered, "derived_binding_mode_disorder_to_order_mobi", {"plddt", "relasa", "sec"})
df_binding_region_do["binding_mode"] = "disorder_to_order"
print(df_binding_region_do.head())
df_binding_region_dd = features_binding_region_df(df_binding_idr_seq_filtered, "derived_binding_mode_disorder_to_disorder_mobi", {"plddt", "relasa", "sec"})
df_binding_region_dd["binding_mode"] = "disorder_to_disorder"
print(df_binding_region_dd.head())
df_binding_region_cd = features_binding_region_df(df_binding_idr_seq_filtered, "derived_binding_mode_context_dependent_mobi", {"plddt", "relasa", "sec"})
df_binding_region_cd["binding_mode"] = "context_dependent"
print(df_binding_region_cd.head())


# save the dataframes as csv files in the processed data folder
df_binding_region_do.to_csv(processed_data_folder / "binding_region_disorder_to_order.csv", index=False)
df_binding_region_dd.to_csv(processed_data_folder / "binding_region_disorder_to_disorder.csv", index=False)
df_binding_region_cd.to_csv(processed_data_folder / "binding_region_context_dependent.csv", index=False)

  uniprot_id                            binding_region_seq  start_pos  \
0     P62258              MDDREDLVYQAKLAEQAERYDEMVESMKKVAG          0   
1     P61981                                FDDAIAELDTLNED        200   
2     Q15172                            QAELHPLPQLKDATSNEQ         49   
3     Q15172                         LFDDLTSSYKAERQREKKKEL        428   
4     Q14738  NSTPPPTQLSKIKYSGGPQIVKKERRQSSSRFNLSKNRELQKLP         60   

   end_pos                                             relasa  \
0       32  [0.8867, 0.5401, 0.5562, 0.1283, 0.5467, 0.320...   
1      214  [0.1491, 0.3636, 0.3476, 0.0, 0.441, 0.6942, 0...   
2       67  [0.7336, 0.4545, 0.5, 0.0366, 0.588, 0.5974, 0...   
3      449  [0.178, 0.0965, 0.5187, 0.492, 0.1675, 0.0982,...   
4      104  [0.8021, 0.8811, 0.7055, 0.7922, 0.7078, 0.798...   

                                               plddt  \
0  [0.5348, 0.8042, 0.9209, 0.9713, 0.97, 0.9722,...   
1  [0.9804, 0.9806, 0.9766, 0.9707, 0.9641, 0.964...   
2 

In [15]:
# combine the three dataframes into one dataframe
df_binding_region_combined = pd.concat([df_binding_region_do, df_binding_region_dd, df_binding_region_cd], ignore_index=True)

df_binding_region_combined["amino_acid_composition"] = df_binding_region_combined["binding_region_seq"].apply(get_amino_acid_composition)
df_binding_region_combined["scd"] = df_binding_region_combined["binding_region_seq"].apply(lambda x: calculate_scd(x, need_rounding=True, round_to=4))
df_binding_region_combined["shd"] = df_binding_region_combined["binding_region_seq"].apply(lambda x: calculate_shd(x, need_rounding=True, round_to=4))
df_binding_region_combined["fcr"] = df_binding_region_combined["binding_region_seq"].apply(lambda x: calculate_fcr(x, need_rounding=True, round_to=4))

print(df_binding_region_combined.head())

TypeError: calculate_scd() got an unexpected keyword argument 'need_rounding'

In [ ]:
# get the unique uniprot ids for each of the three dataframes
#do_uniprot_ids = set(do_data_df["uniprot_id"])
#dd_uniprot_ids = set(dd_data_df["uniprot_id"])
#cd_uniprot_ids = set(cd_data_df["uniprot_id"])

# define the output data folder for the alphasync data
#output_data_folder = current_dir.parents[1] / "data" / "raw" / "alphasync_data"
#output_data_folder = str(output_data_folder)
# download the alphasync data for the unique uniprot ids for each of the three dataframes and save it as csv files in the output data folder
#print("Downloading Alphasync data for disorder to disorder binding mode class...")
#get_alphasync_data(do_uniprot_ids, output_data_folder)
#print("Downloading Alphasync data for disorder to order binding mode class...")
#get_alphasync_data(dd_uniprot_ids, output_data_folder)
#print("Downloading Alphasync data for context dependent binding mode class...")
#get_alphasync_data(cd_uniprot_ids, output_data_folder)


In [ ]:
output_data_folder = current_dir.parents[1] / "data" / "raw" / "alphasync_data"
output_data_folder = str(output_data_folder)


In [5]:
processed_data_folder = current_dir.parents[1] / "data" / "processed"


In [7]:
processed_data_folder = current_dir.parents[1] / "data" / "processed"
do_data_df = pd.read_csv(str(processed_data_folder / "disorder_to_disorder_binding_mode_data.csv"), dtype=str)
do_df_seq = features_binding_region_df(do_data_df, "derived_binding_mode_disorder_to_disorder_mobi", set())
dd_data_df = pd.read_csv(str(processed_data_folder / "disorder_to_order_binding_mode_data.csv"), dtype=str)
dd_df_seq = features_binding_region_df(dd_data_df, "derived_binding_mode_disorder_to_order_mobi", set())
cd_data_df = pd.read_csv(str(processed_data_folder / "context_dependent_binding_mode_data.csv"), dtype=str)
cd_df_seq = features_binding_region_df(cd_data_df, "derived_binding_mode_context_dependent_mobi", set())

In [11]:
do_df_seq["label"] = "disorder_to_disorder"
dd_df_seq["label"] = "disorder_to_order"
cd_df_seq["label"] = "context_dependent"

df_seq = pd.concat([do_df_seq, dd_df_seq, cd_df_seq], ignore_index=True)
df_seq.to_csv(processed_data_folder / "binding_mode_seq_with_labels.csv", index=False)

In [7]:
df_seq = pd.read_csv(processed_data_folder / "binding_mode_seq_with_labels.csv", dtype=str)
df_seq[df_seq["label"] == "disorder_to_disorder"].to_csv(processed_data_folder / "disorder_to_disorder_sequences.csv", index=False, header=False)
df_seq[df_seq["label"] == "disorder_to_order"].to_csv(processed_data_folder / "disorder_to_order_sequences.csv", index=False, header=False)
df_seq[df_seq["label"] == "context_dependent"].to_csv(processed_data_folder / "context_dependent_sequences.csv", index=False, header=False)


In [8]:
df_dd_do_seq = df_seq[df_seq["label"] != "context_dependent"]
df_dd_do_seq.to_csv(processed_data_folder / "disorder_to_disorder_and_disorder_to_order_sequences.csv", index=False, header=False)

In [11]:
df_dd_do_seq

,uniprot_id,binding_region_seq,start_pos,end_pos,label
0,Q04917,QQDEEAGEGN,236,246,disorder_to_disorder
1,P62258,QGDGEEQNKEALQDVEDENQ,235,255,disorder_to_disorder
2,P31947,NAGEEGGEAPQEPQS,233,248,disorder_to_disorder
3,P27348,SAGEECDAAEGAEN,231,245,disorder_to_disorder
4,P63104,QGDEAEAGEGGEN,232,245,disorder_to_disorder
...,...,...,...,...,...
11765,A0A0A0MTJ0,NQKKIRGILESLMCNESS,287,305,disorder_to_order
11766,A0A0A0MTJ0,ETLQAFDSHYDYTICGDSED,375,395,disorder_to_order
11767,U3KQA6,EQELVVTIPKN,183,194,disorder_to_order
11768,A0A140T913,MTHHAVSDHEATL,212,225,disorder_to_order


In [12]:
df_dd_do_seq["id_col"] = df_dd_do_seq["uniprot_id"] + "_" + df_dd_do_seq["start_pos"] + "_" + df_dd_do_seq["end_pos"]

In [13]:
df_dd_do_seq

,uniprot_id,binding_region_seq,start_pos,end_pos,label,id_col
0,Q04917,QQDEEAGEGN,236,246,disorder_to_disorder,Q04917_236_246
1,P62258,QGDGEEQNKEALQDVEDENQ,235,255,disorder_to_disorder,P62258_235_255
2,P31947,NAGEEGGEAPQEPQS,233,248,disorder_to_disorder,P31947_233_248
3,P27348,SAGEECDAAEGAEN,231,245,disorder_to_disorder,P27348_231_245
4,P63104,QGDEAEAGEGGEN,232,245,disorder_to_disorder,P63104_232_245
...,...,...,...,...,...,...
11765,A0A0A0MTJ0,NQKKIRGILESLMCNESS,287,305,disorder_to_order,A0A0A0MTJ0_287_305
11766,A0A0A0MTJ0,ETLQAFDSHYDYTICGDSED,375,395,disorder_to_order,A0A0A0MTJ0_375_395
11767,U3KQA6,EQELVVTIPKN,183,194,disorder_to_order,U3KQA6_183_194
11768,A0A140T913,MTHHAVSDHEATL,212,225,disorder_to_order,A0A140T913_212_225


In [14]:
from df_to_fasta import dataframe_to_fasta
dataframe_to_fasta(df_dd_do_seq, "id_col", "binding_region_seq", "disorder_to_disorder_and_disorder_to_order_sequences.fasta")

In [17]:
# filter same sequences
df_dd_do_seq.drop_duplicates(subset=["binding_region_seq"], inplace=True)
print(f"Number of unique sequences in disorder to disorder and disorder to order binding mode classes: {len(df_dd_do_seq)}")

Number of unique sequences in disorder to disorder and disorder to order binding mode classes: 11762
